# Coffee Data Exploration

Exploring the data extracted from coffeereviews.com

| Date | User | Change Type | Remarks |  
| ---- | ---- | ----------- | ------- |
| 17/03/2026   | Martin | Create  | Notebook created. Creating DuckDB from JSON | 
| 19/03/2026   | Martin | Update  | Updated "Agtron" and "Est. Price" fields due to misnaming. Exploration of document database. ChromaDB checks. | 
| 21/03/2026   | Martin | Update  | Test retrieving column as a list from DuckDB |
| 25/03/2026   | Martin | Update  | Experimentation with LLM finetuning |

# Content

* [DuckDB Ingestion](#duckdb-ingestion)
* [Data Exploration](#data-exploration)

# DuckDB Ingestion

Ingesting raw json data extracted from coffeereviews.com to DuckDB

In [1]:
import json
import duckdb

Check the unique keys in the JSON file

In [11]:
path = "../data/raw"
with open(f"{path}/coffee.json", 'r') as file:
  data = json.load(file)

unique_keys = set()
for item in data:
  if isinstance(item, dict):
    unique_keys.update(item.keys())

list(unique_keys)

['Aroma',
 'Notes',
 'Roast Level',
 'rating',
 'Roaster Location',
 'Flavor',
 'Agtron',
 'Acidity',
 'Who Should Drink It',
 'bean',
 'Explore Similar Coffees',
 'With Milk',
 'Acidity/Structure',
 'Body',
 'Coffee Origin',
 'Aftertaste',
 'Bottom Line',
 'Review Date',
 'Est. Price',
 'Blind Assessment',
 'roaster']

Load into DuckDB @ `/data/warehouse/coffee.duckdb`

In [12]:
JSON_PATH = "../data/raw/coffee.json"
DB_PATH = "../data/warehouse/coffee.duckdb"

con = duckdb.connect(DB_PATH)

# Create the schema
# con.execute("CREATE SCHEMA IF NOT EXISTS coffee;")
con.execute("DROP TABLE IF EXISTS reviews;")

# Data ingestion
con.execute(f"""
  CREATE TABLE reviews AS
    SELECT
      TRY_CAST(data->>'$.rating' AS INTEGER) AS rating,
      (data->>'$.roaster') AS roaster,
      (data->>'$.bean') AS bean,
      (data->>'$.Roaster Location') AS location,
      (data->>'$.Coffee Origin') AS origin,
      (data->>'$.Roast Level') AS roast_level,
      (data->>'$.Agtron') AS agtron,
      (data['Est. Price']) AS est_price,
      strptime(data->>'$.Review Date', '%B %Y')::DATE AS review_date,
      TRY_CAST(data->>'$.Aroma' AS INTEGER) AS aroma,
      TRY_CAST(data->>'$.Body' AS INTEGER) AS body,
      TRY_CAST(data->>'$.Flavor' AS INTEGER) AS flavor,
      TRY_CAST(data->>'$.Aftertaste' AS INTEGER) AS aftertaste,
      TRY_CAST(data->>'$.With Milk' AS INTEGER) AS with_milk,
      TRY_CAST(data->>'$.Acidity/Structure' AS INTEGER) AS acid_structure,
      (data->>'$.Acidity') AS acidity,
      (data->>'$.Blind Assessment')::TEXT AS blind_assessment,
      (data->>'$.Notes')::TEXT AS notes,
      (data->>'$.Who Should Drink It')::TEXT AS who_should_drink,
      (data->>'$.Bottom Line')::TEXT AS bottom_line
    FROM
      read_json('{JSON_PATH}', format='auto') AS data;
""")

In [13]:
con.sql("SHOW ALL TABLES")

┌──────────┬─────────┬─────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────┐
│ database │ schema  │  name   │                                                                                                    column_names                                                                                                     │                                                                                   column_types                                                                                    │ temporary │
│ varchar  │ varchar │ varchar │                                                                                          

In [14]:
con.sql("SELECT * FROM coffee.reviews")

┌────────┬────────────────────────────┬─────────────────────────────────────────────────────────┬───────────────────────────────┬────────────────────────────────────────────────────┬──────────────┬─────────┬────────────────────┬─────────────┬───────┬───────┬────────┬────────────┬───────────┬────────────────┬─────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [11]:
con.close()

# Data Exploration

Exploring the retrieved data

In [2]:
DB_PATH = "../data/warehouse/coffee.duckdb"

con = duckdb.connect(DB_PATH)

In [10]:
# check for missing entries
[item[0] for item in con.sql("SELECT DISTINCT bean FROM coffee.reviews;").fetchall() if item[0] is not None]

['Colombia Andres Martinez Espresso',
 'Citrus Dream Espresso Blend',
 'Ritual Espresso',
 'Yenifer Rojas Trujillo',
 'Ethiopia Guji Hambela Bishan Wate Washed',
 'Eugenioides Rey Coffeea Diversa Costa Rica',
 'Noir',
 'Kayon Mountain Ethiopia Natural',
 'Buesaco Colombia',
 'Mexico Finca Tulipanes',
 'Wilton Benitez Java',
 'Ethiopia Yirgacheffe Banko Gotiti Natural',
 'Ethiopia Washed Yirgacheffe Botabaa G1',
 'Marble City Blend',
 'Ethiopia Sidamo Golden Blossom',
 'Warm Islet Classic Espresso',
 'Hawai’i Kona Geisha Washed Auction Lot',
 'Papua New Guinea Single-Serve K-Cup',
 'Bener Meriah Sumatra Jasmine Co-ferment',
 'Colombia La Esperanza Anaerobic Natural Java',
 'Flight Seasonal Espresso',
 'Ethiopia Sidama Arbegona Solisse 74158 Washed G1',
 'Honduras Bryan Bautista',
 'Ecuador Finca Cahuasqui',
 'Malaysia Liberica Anaerobic Natural',
 'Peru Washed Cusco Buena Vista Soola SL09',
 'Costa Rica La Candelilla Geisha',
 'Ethiopia Fancy',
 'Ethiopia Natural Guji Shakiso Anasora G1

In [7]:
con.close()

# ChromaDB Check

Checking the entries in the ChromaDB.

In [ ]:
import chromadb
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from pprint import pprint

/Users/martz/Repos/bean-juice/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [24]:
client = chromadb.PersistentClient(path="../data/chroma")
collection = client.get_collection(name="coffee")

In [26]:
results = collection.query(
  query_texts=["I like bold and aromatic coffees"],
  n_results=2
)
print(results['documents'][0][0])

The Seattle’s Best Blend is a bean from Seattle, Washington which originated in Indonesia, Central and South America.
    Roasted by Seattle's Best Coffee. 

    OVERALL RATING: 88

    Flavor Profile:
    Roast Level: Medium-Dark
    Agtron: 47/52
    Aroma: 7
    Body: 8
    Flavor: 8
    Aftertaste: 
    With Milk: 
    Acidity/ Structure: 
    Acidity: 6

    Blind Assessment
    This coffee gets better as it goes, probably owing to its heavy body. The dark tones of the aroma modulate with increasing power through the cup to a complex finish.

    Notes
    Coffee with a heavy body that get better as it goes.

    Who Should Drink It
    Classicists who brew with a French press; people who like a cup with power but low acidity; people who love coffee but still pour milk into it. Serve it to business meetings involving people from Boston and Seattle. Neither side may be completely happy, but they won't carp either. Should make an excellent espresso.


In [27]:
con.sql("SELECT * FROM coffee.reviews WHERE bean = 'Seattle’s Best Blend'")

┌────────┬───────────────────────┬──────────────────────┬─────────────────────┬──────────────────────────────────────┬─────────────┬─────────┬───────────┬─────────────┬───────┬───────┬────────┬────────────┬───────────┬────────────────┬─────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────┐
│ rating │        roaster        │         bean         │      location       │                origin                │ roast_level │ agtron  │ est_price │ review_date │ aroma │ body  │ flavor │ afte

In [19]:
embeddings = HuggingFaceEmbeddings(
  model_name="all-MiniLM-L6-v2" ,
  model_kwargs={"device": "cpu"},
  encode_kwargs={"normalize_embeddings": True}
)
vectorstore = Chroma(
  persist_directory="../data/chroma",
  embedding_function=embeddings,
  collection_name="coffee"
)
count = vectorstore._collection.count()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12086.66it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Finetuning

Finetuning a Small LLM to focus on coffee reviews

In [1]:
from huggingface_hub import notebook_login, whoami
notebook_login()
user_info = whoami()
print(user_info['name'])

Minimartzz


In [1]:
import os
import duckdb
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
  BitsAndBytesConfig,
  AutoTokenizer,
  AutoModelForCausalLM,
  TrainingArguments
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer

In [ ]:
DB_PATH = "../data/warehouse/coffee.duckdb"
TRAIN_SPLIT = 0.9

con = duckdb.connect(DB_PATH)
df = con.sql("SELECT * FROM coffee.reviews;").df()

# Remove entries where both bottom_line and who_should_drink are empty (target col)
mask = df[(df['bottom_line'].isna()) & (df['who_should_drink'].isna())].index
df = df[~df.index.isin(mask)]

# Merge both bottom_line and who_should drink
df['recommendation'] = df['bottom_line'].str.cat(df['who_should_drink'], sep=' ', na_rep='')

df.head()

,rating,roaster,bean,location,origin,roast_level,agtron,est_price,review_date,aroma,...,flavor,aftertaste,with_milk,acid_structure,acidity,blind_assessment,notes,who_should_drink,bottom_line,recommendation
0,96,Paradise Roasters,Kona SL34 Peaberry,"Hilo, Hawai’i Island, Hawai’i","Kona growing region, Hawai’i Island, Hawai'i",Medium-Light,58/78,$50.00/4 ounces,2026-03-01,9,...,10,9,<NA>,9,NaN,"Bright, juicy, vibrantly sweet. Passionfruit, ...","Produced by Kraig Lee of Kona Farm Direct, ent...",NaN,A beautifully composed SL34 grown in Kona that...,A beautifully composed SL34 grown in Kona that...
1,96,JBC Coffee Roasters,El Proceso Chiroso,"Madison, Wisconsin","Tolima Department, Colombia",Medium-Light,57/78,$26.00/8 ounces,2026-03-01,9,...,10,9,<NA>,9,NaN,"Balanced, richly aromatic, berry-driven. Black...",Produced by cousins Stiven Vargas Rojas and Jo...,NaN,A beautifully articulated Chiroso that marries...,A beautifully articulated Chiroso that marries...
2,92,Merge Coffee Company,Dominican Republic Ramirez Estate,"Harrisonburg, Virginia","Jarabacoa, Dominican Republic",Medium-Light,52/72,$19.99/12 ounces,2026-03-01,8,...,9,8,<NA>,8,NaN,"Crisply sweet, berry-toned. Raspberry, baking ...",Produced at Ramirez Estate and processed by th...,NaN,A richly satisfying Dominican natural that bal...,A richly satisfying Dominican natural that bal...
3,93,Merge Coffee Company,Ethiopia Sweet Lily,"Harrisonburg, Virginia",Ethiopia,Medium-Light,58/77,$18.99/12 ounces,2026-03-01,9,...,9,8,<NA>,9,NaN,"Brightly floral, sweetly citrus-toned. Lemon b...",A blend of coffees from multiple regions in Et...,NaN,A solid washed Ethiopia that highlights floral...,A solid washed Ethiopia that highlights floral...
4,93,Big Shoulders Coffee,Colombia El Espejo,"Chicago, Illinois","Planadas, Tolima Department, Colombia",Light,63/89,$24.00/12 ounces,2026-03-01,8,...,9,8,<NA>,9,NaN,"Sweetly bright, gently fruit-toned. Red apple,...","Produced by Reinel Borbon of Finca El Espejo, ...",NaN,A balanced honey-processed Colombia Tabi (a hy...,A balanced honey-processed Colombia Tabi (a hy...


1. Build the text prompt for model to learn

In [2]:
BASE_MODEL  = "Qwen/Qwen3-1.7B" 
DB_PATH     = "../data/warehouse/coffee.duckdb"
OUTPUT_DIR  = "../models/coffee-lora"
MAX_SEQ_LEN = 512
TRAIN_SPLIT = 0.9

In [3]:
def build_prompt(row: dict, inference: bool = False) -> str:
  """
  Converts a row of data into a structured prompt. During inference, omit the answer, so the model can generate it
  """
  def fmt(val, label):
    return f"{label}: {int(val)}/10" if pd.notna(val) else ""
  
  score_parts = [
    fmt(row.get("aroma"),          "Aroma"),
    fmt(row.get("body"),           "Body"),
    fmt(row.get("flavor"),         "Flavor"),
    fmt(row.get("aftertaste"),     "Aftertaste"),
    fmt(row.get("acid_structure"), "Acid Structure"),
  ]
  scores = " | ".join(p for p in score_parts if p)

  def safe(val):
    return str(val).strip() if pd.notna(val) else ""

  prompt = f"""### Coffee Review
Roaster: {safe(row.get('roaster'))}
Bean: {safe(row.get('bean'))}
Origin: {safe(row.get('origin'))}
Roast Level: {safe(row.get('roast_level'))}
Rating: {row.get('rating', 'N/A')}/100
Scores: {scores}
Price: {safe(row.get('est_price'))}

Blind Assessment:
{safe(row.get('blind_assessment'))}

Notes:
{safe(row.get('notes'))}

Write a concise recommendation for this coffee.
"""
  messages = [
    {"role": "system", "content": "You are a coffee expert who writes concise, accurate recommendations."},
    {"role": "user", "content": prompt}
  ]
  if not inference:
    messages.append({"role": "assistant", "content": safe(row.get("recommendation"))})
  
  return tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=inference
  )

2. Load the data

In [4]:
def load_data(path: str, tokenizer):
  """
  Load duckdb database and transform the data using the prompt format
  """
  con = duckdb.connect(path)
  df = con.sql("SELECT * FROM coffee.reviews;").df()

  # Remove entries where both bottom_line and who_should_drink are empty (target col)
  mask = df[(df['bottom_line'].isna()) & (df['who_should_drink'].isna())].index
  df = df[~df.index.isin(mask)]

  # Merge both bottom_line and who_should drink
  df['recommendation'] = df['bottom_line'].str.cat(df['who_should_drink'], sep=' ', na_rep='')

  # Build prompt
  df['text'] = df.apply(lambda r: build_prompt(r.dict() if hasattr(r, "dict") else r.to_dict(), tokenizer), axis=1)

  # Train/ Val split
  df = df.sample(frac=1, random_state=43).reset_index(drop=True)
  split = int(len(df) * TRAIN_SPLIT)
  df_train = df.iloc[:split]
  df_val   = df.iloc[split:]

  train_ds = Dataset.from_pandas(df_train[['text']])
  val_ds   = Dataset.from_pandas(df_val[['text']])

  print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")
  return train_ds, val_ds, df_val

3. Load model and tokenizer

In [5]:
def load_model_and_tokenizer(model_name: str):
  # 4-bit quantization config (QLoRA)
  bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
  )

  # Tokenizer
  tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
  tokenizer.pad_token = tokenizer.eos_token
  tokenizer.padding_side = "right"

  # Model
  model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
  )
  model.config.use_cache = False

  return model, tokenizer

4. Define the LoRA configuration

In [6]:
def apply_lora(model):
  lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16, # rank - higher = higher attention dimensions, more capacity to learn, more memory req
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
  )

  model = get_peft_model(model, lora_config)
  model.print_trainable_parameters()
  return model

5. Define the training loop

In [7]:
def train(model, tokenizer, train_ds, val_ds):

  def tokenize(batch):
    return tokenizer(
      batch["text"],
      truncation=True,
      max_length=MAX_SEQ_LEN,
      padding="max_length",
    )

  train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])
  val_ds = val_ds.map(tokenize, batched=True, remove_columns=["text"])

  training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    warmup_ratio=0.05,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    optim="paged_adamw_32bit",
    gradient_checkpointing=True,
  )

  trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
  )

  trainer.train()
  trainer.save_model(OUTPUT_DIR)
  tokenizer.save_pretrained(OUTPUT_DIR)
  print(f"\nModel saved to {OUTPUT_DIR}")
  return trainer

6. Inference generation

In [8]:
def generate_recommendation(model, tokenizer, row: dict, max_new_tokens=150) -> str:
  prompt = build_prompt(row, inference=True)
  inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

  with torch.no_grad():
    outputs = model.generate(
      **inputs,
      max_new_tokens=max_new_tokens,
      temperature=0.7,
      do_sample=True,
      top_p=0.9,
      eos_token_id=tokenizer.eos_token_id,
      pad_token_id=tokenizer.pad_token_id
    )
  
  # Decode only the newly generated tokens
  generated = outputs[0][inputs["input_ids"].shape[1]:]
  return tokenizer.decode(generated, skip_special_tokens=True).strip()

Main Training Loop

In [9]:
print("=" * 30)
print("Loading Model")
print("=" * 30)
model, tokenizer = load_model_and_tokenizer(BASE_MODEL)

print("=" * 30)
print("Loading Data")
print("=" * 30)
train_ds, val_ds, val_df = load_data(DB_PATH, tokenizer)

print("=" * 30)
print("Applying LoRA")
print("=" * 30)
model = apply_lora(model)

print("=" * 30)
print("Training")
print("=" * 30)
trainer = train(model, tokenizer, train_ds, val_ds)

print("=" * 30)
print("Sample Generation")
print("=" * 30)
sample = val_df.iloc[0].to_dict()
print("Expected:", sample["recommendation"])
print("Generated:", generate_recommendation(model, tokenizer, sample))

Loading Model


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading Data
Train: 7973 | Val: 886
Applying LoRA
trainable params: 17,432,576 || all params: 2,049,172,480 || trainable%: 0.8507
Training


Map:   0%|          | 0/7973 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/886 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.
/Users/martz/Repos/bean-juice/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


W0325 17:44:09.800000 17190 torch/_inductor/codegen/mps.py:188] [0/5_1] float64 cast requested, probably from tensorify_python_scalars
W0325 17:44:09.805000 17190 torch/_inductor/codegen/mps.py:188] [0/5_1] float64 cast requested, probably from tensorify_python_scalars
W0325 17:44:09.807000 17190 torch/_inductor/codegen/mps.py:188] [0/5_1] float64 cast requested, probably from tensorify_python_scalars
W0325 17:44:09.966000 17190 torch/_inductor/codegen/mps.py:188] [0/6] float64 cast requested, probably from tensorify_python_scalars
W0325 17:44:09.971000 17190 torch/_inductor/codegen/mps.py:188] [0/6] float64 cast requested, probably from tensorify_python_scalars
W0325 17:44:09.972000 17190 torch/_inductor/codegen/mps.py:188] [0/6] float64 cast requested, probably from tensorify_python_scalars
W0325 17:44:10.070000 17190 torch/_inductor/codegen/mps.py:188] [0/7] float64 cast requested, probably from tensorify_python_scalars
W0325 17:44:10.075000 17190 torch/_inductor/codegen/mps.py:188]

KeyboardInterrupt: 

In [2]:
%watermark

Last updated: 2025-06-18T19:03:45.452311+08:00

Python implementation: CPython
Python version       : 3.11.9
IPython version      : 8.31.0

Compiler    : MSC v.1938 64 bit (AMD64)
OS          : Windows
Release     : 10
Machine     : AMD64
Processor   : Intel64 Family 6 Model 183 Stepping 1, GenuineIntel
CPU cores   : 20
Architecture: 64bit

